# 06c: 亚群 Subset 重分析

面向 **非计算机专业 PI/学生**（ADR-0009）。

当 PI 在 06 全局注释后希望对某类细胞（如所有 T 细胞 / 上皮细胞 /
SPEM 谱系）做更精细的亚群分析时，本 notebook 执行以下流程：

1. **子集抽取**：按 `SUBSET_FILTER` 表达式筛选（如 `cell_type_final_v1.isin(['CD4 T', 'CD8 T', 'Treg'])`）
2. **03 重跑**：对子集重新选 HVG（全局 HVG 对亚群未必最优）
3. **04 重跑**：对子集重新做嵌入（子集的批次效应/生物变异结构不同）
4. **05 重跑**：对子集重新聚类（子集通常需要更细的分辨率）
5. **06 重跑**：对子集重新标注（精细细胞亚型，如 T_cell → CD4_Tcm/CD8_Tem）
6. **标签回流**：将精细标签写回主图谱的新列 `cell_type_final_subset_v1`——
   **子集内的细胞**获得精细标签，**子集外的细胞保留 NaN**。
   原 `cell_type_final_v1` 列**不覆盖**——保留层级粒度（粗在 `_final_v1`，细在 `_final_subset_v1`）。

**为什么需要亚群重分析？**
全局分析的目标是区分大类（上皮 vs 免疫 vs 间质），参数为这个目标校准。
但同一大类内部的异质性（如 T 细胞的 CD4/CD8 亚群、上皮的 pit/neck/SPEM 梯度）
需要重新选 HVG + 重新聚类才能看清——这是 scRNA-seq 分析的标准实践，
不是"重跑浪费算力"。

**溯源机制**：`adata_sub.uns["subset_of"]` 记录来源 h5ad，
`adata_sub.uns["subset_filter"]` 记录筛选表达式，
任何下游分析都能追溯这个子集是如何产生的（见 SPEC 196）。

**实现纪律（ADR-0003/0009）**：直接调 scanpy 原生 API，
无 plugin/registry/class。每个 stage 是独立的 cell block，
非 CS 学生按顺序读下来能看懂每一步在做什么。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：06（注释），读 `06_annotated_v*.h5ad`
- **下游**：07（下游分析），产出两个文件：
  - `06c_subset_v*.h5ad`（子集独立 h5ad）
  - `06_annotated_v*.h5ad`（主 adata 回流更新版，含 `cell_type_final_subset_v1` 列）

### 为什么要迭代回跑？
亚群重分析的质量取决于上游注释的准确性 + 子集筛选的合理性 + 子集内参数的选择。
当以下任一情况发生时需要重跑：
- PI 发现子集注释不够精细（如 T 细胞应拆成 CD4/CD8/Treg 但未拆）
- PI 调整了全局的 `cell_type_final_v1` 标签（上游 06 重跑后）
- PI 想分析另一类细胞（改 `SUBSET_FILTER`）
- 子集 Leiden 聚类结果不理想（改 `RESOLUTIONS` 或 `N_PCS`）
- 标记物 CSV 更新（改 `MARKER_CSV`）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_PATH`**——指向要复用的上游文件版本
   （例如 `06_annotated_v2.h5ad`）
2. **改 `OUTPUT_PATH`**——bump 子集版本号 `_v1` → `_v2`
   （例如 `06c_subset_v2.h5ad`）
   同时改 `MAIN_OUTPUT_PATH`——bump 主 adata 版本号
   （例如 `06_annotated_v3.h5ad`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改 `SUBSET_FILTER`、`N_PCS`、
   `RESOLUTIONS` 或标记物参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`_v1` / `_v2` / ...**：每次调参重跑 bump 一位版本号。
  旧版 `.h5ad` 文件**不覆盖不删除**，保留在 `results/` 目录供追溯对比。
- **`experimental`**：刚跑出、尚未经 PI 审查确认的版本（默认值）。
- **`promoted`**：PI 审查后认为子集分析质量可接受、可传给下游使用的正式版本。

### 两个输出对象的溯源策略（重要）
本 notebook 产出两个 h5ad，但**区别对待**：
- **`adata_sub`（子集对象）**：全新对象，独立顶层追踪字段
  （`stage="06c_subset"`、`version="v1"`、`upstream=[UPSTREAM_PATH]`、`status="experimental"`）。
  保留其 `06c_subset_v1` 嵌套 dict 作为细节记录。
- **`main_adata`（主对象回流更新）**：本身是 06 的产物，**不覆盖**
  其顶层 `stage` / `version` / `upstream` 字段——那会篡改 06 的溯源链。
  `main_adata` 仅写入 `06c_subset_reflow_v1` 嵌套 dict 记录本次回流操作。
  主对象的追踪字段由 06 负责维护。

### 追溯链
如需查询"06c 有哪些版本？"或"这个子集依赖哪个 06 版本？"，
可在 Python 中检查子集 h5ad 的 `adata.uns["stage"]` / `adata.uns["upstream"]`。


In [ ]:
# === PARAMS ===
# UPSTREAM_PATH       -- 06_annotated 输出 h5ad（必须有 cell_type_final_v1）
# SUBSET_FILTER       -- 筛选表达式（Python 表达式，作用在 adata.obs 上）
# OUTPUT_PATH         -- 子集 h5ad 版本化输出（06c 产物）。
#                        版本号 _v1 与 adata_sub.uns["version"] 保持一致。
#                        如需回跑：bump 版本号 _v1->v2，旧版不覆盖。
# MAIN_OUTPUT_PATH    -- 主 adata 更新后的输出（含 cell_type_final_subset_v1 列）
# N_TOP_GENES         -- HVG 数量
# N_PCS               -- PCA 主成分数
# N_PCS_USE           -- 实际送入邻居图/Harmony 的 PC 数（与 04/05 一致）
# RESOLUTIONS         -- Leiden 多分辨率列表
# MARKER_CSV          -- 标记物知识库 CSV
# RANDOM_SEED         -- 随机种子（全流程一致）

UPSTREAM_PATH  = "results/06_annotated_v1.h5ad"
SUBSET_FILTER  = "cell_type_final_v1.isin(['CD4 T', 'CD8 T', 'Treg', 'NK cell', 'B cell'])"
OUTPUT_PATH    = "results/06c_subset_v1.h5ad"

# 主 adata 更新输出（版本号随着每次 subset 回流递增）
MAIN_OUTPUT_PATH = "results/06_annotated_v2.h5ad"

# ---- 分析参数 ----
N_TOP_GENES = 3000
N_PCS       = 50
N_PCS_USE   = 30  # 实际送入邻居图/Harmony 的 PC 数（与 04/05 一致）
RESOLUTIONS = [0.4, 0.6, 0.8, 1.0, 1.2, 1.6]
MARKER_CSV  = "references/markers/gastric_TEST_markers.csv"

# ---- LLM（统一 llm_config 路由，从 .env LLM_GROUP* 读取）----
MLLM_ENABLED = True
MLLM_MODELS = None          # None = 从 .env 自动构建；手动指定如 ["claude-3-5-haiku-20241022"]
MLLM_CONSENSUS_THRESHOLD = 0.7
MLLM_ENTROPY_THRESHOLD = 0.5   # 子集模式簇数少、标签空间窄，模型分歧天然更大，比全局 0.3 适当放宽
MLLM_MAX_DISCUSSION_ROUNDS = 3

RANDOM_SEED = 42

# 输出列名版本号——集中管理（与 06_annotated 的 OUTPUT_VERSION 一致）
# 默认 "v1" 与现有下游兼容；bump 时需同步更新 MAIN_OUTPUT_PATH 中的版本号
OUTPUT_VERSION = "v1"


# 每运行一次 06c 递增 N
# 如 T cells 做了一次，上皮又做了一次 → N 分别 = 1, 2

In [ ]:
# 确保框架 src/ 在 sys.path 并切换到项目根目录。
import sys, os, gc, datetime, warnings
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures/06c_subset", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 导入（scanpy 原生 + 框架函数）
import scanpy as sc
import scvi
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from scrna_integration import load_markers

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
warnings.filterwarnings("ignore", category=FutureWarning)

# 固定随机种子（全流程一致，保证可复现）
np.random.seed(RANDOM_SEED)

# 加载上游 adata
print(f"加载上游: {UPSTREAM_PATH}")
adata = sc.read_h5ad(UPSTREAM_PATH)
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")

## 子集抽取

按 `SUBSET_FILTER` 表达式筛选细胞，`copy()` 保证子集独立于主 adata。
在子集 `uns` 中记录 `subset_of` 和 `subset_filter` 以供溯源。

**为什么用 `.copy()` 而非 view？**
后面的 HVG 重选、归一化、嵌入都需要独立的内存空间。
view 会意外修改主 adata 的数据，copy 隔离这个风险。

**为什么先确认 `cell_type_final_v1` 存在？**
subset 筛选依赖于这个列——如果该列不存在（如 PI 还没拍板），
整条分析链路无意义。提前报错比后面发现好。

In [ ]:
# 确认 cell_type_final_v1 存在（subset 筛选依赖它）。
if "cell_type_final_v1" not in adata.obs.columns:
    raise KeyError(
        "obs 中没有 'cell_type_final_v1' 列。"
        "请先运行 06_annotated.ipynb 并完成 PI 拍板（pi_decisions）。"
    )

# 执行子集筛选——为什么用 eval？让 PI 在 PARAMS 中写人类可读的表达式，
# 而非在代码里硬编码标签列表。
print(f"筛选表达式: {SUBSET_FILTER}")
_mask = adata.obs.eval(SUBSET_FILTER)
print(f"筛选前: {adata.n_obs:,} 细胞")
print(f"筛选后: {_mask.sum():,} 细胞 ({_mask.sum()/adata.n_obs*100:.1f}%)")

if _mask.sum() < 50:
    raise ValueError(
        f"子集仅 {_mask.sum()} 细胞，不足以做有意义的亚群分析。"
        "请检查 SUBSET_FILTER。"
    )

# copy() 保证独立内存空间
adata_sub = adata[_mask].copy()
print(f"\n子集 adata: {adata_sub.n_obs:,} 细胞 x {adata_sub.n_vars:,} 基因")

# ---- 溯源元数据 ----
# 为什么记录这些？任何下游分析或合作者拿到这个 h5ad 后，
# 通过 uns["subset_of"] 和 uns["subset_filter"] 就能追溯来源——
# 不需要翻 notebook 找参数。
adata_sub.uns["subset_of"] = UPSTREAM_PATH
adata_sub.uns["subset_filter"] = SUBSET_FILTER
adata_sub.uns["subset_n_cells_before"] = adata.n_obs
adata_sub.uns["subset_n_cells_after"] = adata_sub.n_obs
print(f"溯源信息已记录: subset_of={UPSTREAM_PATH}")
print(f"                   subset_filter={SUBSET_FILTER}")

# 释放主 adata（后续不需要它，直到回流步骤）
del adata
gc.collect()
print("主 adata 已释放")

## 03: 归一化 + 高变基因重选

**为什么子集要重选 HVG？**
全局 HVG 选的是能区分所有大类（上皮 vs 免疫 vs 间质）的基因。
但在 T 细胞子集中，这些基因大部分不表达（如上皮特异基因 MUC5AC），
真正区分 CD4/CD8/Treg 的基因（如 CD4/CD8A/FOXP3）在全局 HVG 中
可能因为跨大类差异不够大而被过滤掉。
因此子集分析的第一步永远是 **对子集重新选 HVG**。

**为什么先存 counts 层？**
归一化会覆盖 `adata.X`，但 06 的基因集评分等下游可能需要 raw counts。
存到 `layers["counts"]` 是 scanpy 标准实践，学生应该学会这个习惯。

In [ ]:
# 03: 归一化 + log + HVG（在子集上重跑）。
print("=== 03: normalize + HVG re-selection on subset ===")

# 保留 raw counts（归一化前）
# 为什么存 layers["counts"]？这是 scanpy 社区约定——
# 任何下游分析需要 raw counts 时从这里取，不用回头找 02 输出。
adata_sub.layers["counts"] = adata_sub.X.copy()

# 归一化到 10,000 counts per cell
# 为什么 target_sum=1e4？这是 scRNA-seq 的标准归一化目标——
# 与 10x Genomics 的默认值、Cell Ranger 的输出口径一致。
sc.pp.normalize_total(adata_sub, target_sum=1e4)
sc.pp.log1p(adata_sub)

# HVG 重选——用 seurat_v3 flavor
# 为什么 flavor="seurat_v3"？seurat_v3 基于方差稳定化变换选 HVG，
# 对比默认的 seurat（基于 dispersion），在子集分析中对稀有亚群的标记
# 基因检出率更高。见 ADR-0009 注释中文化时对 flavor 选择的讨论。
sc.pp.highly_variable_genes(
    adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
)
# 也可以额外指定 batch_key 避免批次驱动 HVG 选择：
# sc.pp.highly_variable_genes(
#     adata_sub, n_top_genes=N_TOP_GENES, flavor="seurat_v3",
#     batch_key="source_dataset",
# )

n_hvg = adata_sub.var["highly_variable"].sum()
print(f"\nHVG 选择: {n_hvg}/{adata_sub.n_vars} 基因标记为 highly_variable")
print(f"  前 10 个 HVG: {list(adata_sub.var_names[adata_sub.var['highly_variable']][:10])}")

# 可视化检查
sc.pl.highly_variable_genes(adata_sub, show=False)
plt.savefig("results/figures/06c_subset_hvg.png", dpi=120, bbox_inches="tight")
plt.close()
print("  HVG 图已保存: results/figures/06c_subset_hvg.png")

# 内存纪律：归一化不改变 sparse 性质，但确认 dtype
adata_sub.X = adata_sub.X.astype(np.float32)
assert sp.issparse(adata_sub.X) and adata_sub.X.dtype == np.float32
print("  内存自检: X sparse CSR float32 OK")

# 记录运行元数据
adata_sub.uns["normalize_v1"] = {
    "target_sum": 1e4,
    "hvg_flavor": "seurat_v3",
    "n_top_genes": N_TOP_GENES,
    "n_hvg": int(n_hvg),
    "subset": True,  # 标记这是子集重跑
    "timestamp": datetime.datetime.now().isoformat(),
}

## 04: 多方法嵌入（在子集上）

与全局 04 相同的方法阵容，但在子集上独立运行。

**为什么子集需要重新嵌入？**
全局 PCA 的 loadings 由所有细胞的方差结构决定——在上皮细胞主导的
全局数据中，PC1-3 大概率是上皮 vs 免疫的差异。T 细胞子集的内部
变异（CD4 vs CD8、naive vs memory）在这些 PC 上可能是噪声。
在子集上重新跑 PCA+整合，嵌入空间才能真正反映子集内部的生物学变异。

**批次整合**：子集通常跨多个 source_dataset（病种/平台），
仍需要 Harmony/scVI 去批次。使用与全局相同的 `batch_key="source_dataset"`。

In [ ]:
# 04: PCA + Harmony + scVI（在子集上独立运行）。
print("=== 04: embedding on subset ===")

# ---- PCA ----
sc.tl.pca(adata_sub, n_comps=N_PCS, use_highly_variable=True, svd_solver="arpack")
print(f"PCA 完成: {N_PCS} PCs")

# ---- Harmony 去批次（harmonypy 直调，避免 scanpy wrapper >=2.0 维度 bug）----
# 为什么 batch_key="source_dataset"？不同数据集可能存在不同的技术/平台
# 批效应。Harmony 在 PCA 空间做 soft-clustering 对齐，
# 对 scRNA-seq 的稀疏数据比 MNN/CCA 更快且不易过校正。
# 为什么直调 harmonypy？sc.external.pp.harmony_integrate 在 harmonypy >=2.0
# 有维度兼容问题——直调避免中间层版本耦合。
import harmonypy

ho = harmonypy.run_harmony(
    adata_sub.obsm["X_pca"][:, :N_PCS_USE],  # 只用前 N_PCS_USE 个 PC
    adata_sub.obs,
    "source_dataset",
    max_iter_harmony=20,
    random_state=RANDOM_SEED,
)
adata_sub.obsm["X_pca_harmony"] = ho.Z_corr
assert ho.Z_corr.shape == (adata_sub.n_obs, N_PCS_USE), (
    f"Harmony shape 不符: {ho.Z_corr.shape} != ({adata_sub.n_obs}, {N_PCS_USE})")
print(f"✓ Harmony 完成: {adata_sub.obsm['X_pca_harmony'].shape}")

# 收敛检查
if hasattr(ho, 'check_convergence'):
    _converged = ho.check_convergence()
elif hasattr(ho, 'converged'):
    _c = ho.converged
    _converged = _c() if callable(_c) else _c
else:
    _converged = True

if not _converged:
    print(f"⚠️ Harmony 未收敛——考虑增大 max_iter 或检查子集内批次结构")

# ---- scVI（深度生成模型）----
# scVI 假设计数数据服从零膨胀负二项分布，从 raw counts 学习隐变量。
# 为什么用 counts 层而非 normalized X？scVI 内部有自己的归一化机制，
# 直接用 counts 避免丢失分布信息。
scvi.model.SCVI.setup_anndata(
    adata_sub, batch_key="source_dataset", layer="counts",
)
_model = scvi.model.SCVI(adata_sub, n_latent=30, n_layers=2)
# max_epochs 可根据子集大小调整——子集小（<5000 cells）用较少 epoch
_max_epochs = 200 if adata_sub.n_obs > 5000 else 100
_model.train(max_epochs=_max_epochs, early_stopping=True)
adata_sub.obsm["X_scVI"] = _model.get_latent_representation()
print(f"scVI 完成: obsm['X_scVI'] shape={adata_sub.obsm['X_scVI'].shape}")

# 记录元数据
adata_sub.uns["embedding_v1"] = {
    "methods": ["pca", "harmony", "scvi"],
    "n_pcs": N_PCS,
    "n_latent": 30,
    "batch_key": "source_dataset",
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}
print("  嵌入元数据已记录")

## 05: 多分辨率聚类（在子集上）

用 Harmony 嵌入构建 kNN 图，然后做多分辨率 Leiden 聚类。
**为什么用 Harmony embedding 建图？** Harmony 已去除已知批次效应，
在此基础上建图能减少"同细胞类型因技术差异被拆成不同簇"的问题。

**为什么在子集上用更细的分辨率（1.0–1.6）？**
子集中的生物学差异比全局更 subtle——CD4 Tcm vs CD4 Tem 之间的
标记基因差异远小于 T cell vs epithelial 之间的差异。
更细的分辨率让 Leiden 能捕捉这些微妙结构。

PI 在多个分辨率中选最合理的，方法与全局 05 相同。

In [ ]:
# 05: 多分辨率 Leiden 聚类（在子集上）。
print("=== 05: clustering on subset ===")

# 用 Harmony 嵌入构建 kNN 图
# 为什么 use_rep="X_pca_harmony"？去批次后再建邻居关系，
# 避免"相同生物学但因批次差异被归为不同簇"。
sc.pp.neighbors(adata_sub, use_rep="X_pca_harmony", n_neighbors=15,
                n_pcs=N_PCS_USE, random_state=RANDOM_SEED)

# 计算 UMAP（供后续可视化）
sc.tl.umap(adata_sub)
print(f"UMAP 完成: obsm['X_umap']")

# 多分辨率 Leiden clustering
for res in RESOLUTIONS:
    key = f"leiden_res_{res}"
    sc.tl.leiden(adata_sub, resolution=res, key_added=key)
    n_clusters = adata_sub.obs[key].nunique()
    print(f"  leiden_res_{res}: {n_clusters} 簇")

print(f"\n多分辨率聚类完成。{len(RESOLUTIONS)} 个分辨率已写入 obs。")

# 可视化各分辨率
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
axes = axes.flatten()
for ax, res in zip(axes, RESOLUTIONS):
    sc.pl.umap(
        adata_sub, color=f"leiden_res_{res}", ax=ax,
        title=f"Leiden res={res}", legend_loc="right margin",
        show=False,
    )
for ax in axes[len(RESOLUTIONS):]:
    ax.set_visible(False)
fig.tight_layout()
fig.savefig("results/figures/06c_subset_leiden_sweep.png", dpi=120, bbox_inches="tight")
plt.close(fig)
print("  Leiden sweep 图已保存: results/figures/06c_subset_leiden_sweep.png")

# 记录元数据
adata_sub.uns["clustering_v1"] = {
    "use_rep": "X_pca_harmony",
    "resolutions": RESOLUTIONS,
    "n_neighbors": 15,
    "subset": True,
    "timestamp": datetime.datetime.now().isoformat(),
}

### PI 选择子集聚类分辨率

多分辨率中 PI 选择一个最合理的分辨率作为本子集的标签列。
通常的原则是：簇数不能太多（过度分裂）也不能太少（欠聚类），
要能在 UMAP 上看到清晰的群体分离。

In [ ]:
# === PI 选择子集聚类分辨率 ===
# PI 查看上方的 Leiden sweep UMAP 图后，选择一个分辨率。
# 为什么让 PI 选？自动选分辨率的算法（如 silhouette 最大值）
# 对子集分析的 subtle 结构不够敏感——PI 的领域知识在这里不可替代。
LEIDEN_COL = "leiden_res_0.6"  # PI 从上方图中选一个最合理的分辨率

if LEIDEN_COL not in adata_sub.obs.columns:
    print(f"⚠ 警告: '{LEIDEN_COL}' 不在 obs 中。可用列: "
          f"{[c for c in adata_sub.obs.columns if c.startswith('leiden_')]}")
else:
    n_clust = adata_sub.obs[LEIDEN_COL].nunique()
    print(f"选用聚类列: {LEIDEN_COL} ({n_clust} 簇)")
    # 确保是 categorical
    if not pd.api.types.is_categorical_dtype(adata_sub.obs[LEIDEN_COL]):
        adata_sub.obs[LEIDEN_COL] = adata_sub.obs[LEIDEN_COL].astype("category")

## 06: 子集重标注（精细细胞亚型）

与全局 06 相同的多方法标注流程，但目标改为精细亚型：
- 全局 06 标注的是：T_cell, B_cell, pit_cell, SPEM...
- 子集 06 标注的是：CD4_Tcm, CD8_Tem, Treg, MAIT...（在 T 细胞子集内）

**为什么用多方法？**
与全局 06 相同的理由——单方法容易出错，多方法交叉比对才能
找到置信度高的标签。但这里调用 mLLMCelltype 时，tissue prompt
应调整为子集对应的 context（如 "human peripheral blood T cells"
而非 "human stomach"）。

**注意**：本 cell 的 LLM 调用同样受 key 守卫——无 key 时优雅跳过。

In [ ]:
# 06: 子集重标注（精细细胞亚型）。
print("=== 06: re-annotation on subset (finer cell types) ===")

# ---- B4：应用 mLLMCelltype monkey-patch（与 06 统一，修复库已知缺陷） ----
from scrna_integration.llm_config import (
    apply_mllmcelltype_patches,
    build_mllmcelltype_config,
)
_patch_ok = apply_mllmcelltype_patches(
    max_retries=3,
    retry_delay=2,
    timeout=120,
    max_tokens_override=16384,
)
if _patch_ok:
    print("mLLMCelltype patch 已应用")
else:
    print("\u26a0\ufe0f mLLMCelltype patch 失败，LLM 注释将跳过")

# ---- 方法 1: 标记物 dotplot（PI 手动标注）----
try:
    _markers = load_markers(MARKER_CSV)
    print(f"\u2713 标记物库加载: {MARKER_CSV} ({sum(len(v) for v in _markers.values())} 基因)")
except FileNotFoundError:
    print(f"\u26a0\ufe0f 标记物文件不存在: {MARKER_CSV}——跳过 dotplot 和基因集评分")
    _markers = {}

# === Marker 库与 subset 类型一致性检查 ===
# 检测 SUBSET_FILTER 是否为免疫/上皮类型，与 MARKER_CSV 做一致性检查

_marker_filename = os.path.basename(MARKER_CSV).lower()

# B4：从子集筛选表达式推导 tissue context（代替硬编码 "human gastric mucosa"）
_immune_keywords = {"immune", "t_cell", "b_cell", "cd4", "cd8", "treg", "nk", "myeloid",
                    "t cell", "b cell", "nk cell"}
_epithelial_keywords = {"epithelial", "chief", "parietal", "mucous", "spem", "pit"}

_filter_lower = SUBSET_FILTER.lower()
_filter_is_immune = any(kw in _filter_lower for kw in _immune_keywords)
_filter_is_epithelial = any(kw in _filter_lower for kw in _epithelial_keywords)

if _filter_is_immune:
    _tissue_context = "human immune cells"
    print(f"子集组织上下文: {_tissue_context}（从 SUBSET_FILTER 免疫关键词推导）")
elif _filter_is_epithelial:
    _tissue_context = "human gastric epithelium"
    print(f"子集组织上下文: {_tissue_context}（从 SUBSET_FILTER 上皮关键词推导）")
else:
    _tissue_context = "human gastric mucosa"
    print(f"子集组织上下文: {_tissue_context}（默认，无法从 SUBSET_FILTER 推导）")

if _filter_is_immune and "epithelial" in _marker_filename:
    print("\u26a0\ufe0f SUBSET_FILTER 选择的是免疫细胞，但 MARKER_CSV 是上皮标记物！")
    print(f"   SUBSET_FILTER: {SUBSET_FILTER}")
    print(f"   MARKER_CSV: {MARKER_CSV}")
    print("   -> 建议改为免疫标记物 CSV（如 gastric_immune.csv），否则 dotplot/评分无意义")
elif _filter_is_epithelial and "immune" in _marker_filename:
    print("\u26a0\ufe0f SUBSET_FILTER 选择的是上皮细胞，但 MARKER_CSV 是免疫标记物！")
    print(f"   -> 建议改为上皮标记物 CSV")
else:
    print(f"Marker 库与 subset 类型一致性检查通过")
    print(f"   SUBSET_FILTER 关键词: {_filter_lower[:80]}...")
    print(f"   MARKER_CSV: {MARKER_CSV}")

_all_marker_genes = sorted(set(g for glist in _markers.values() for g in glist))
_avail = [g for g in _all_marker_genes if g in adata_sub.var_names]
_avail = _avail[:30]  # 子集分析减少展示基因数避免图过大

if _avail and LEIDEN_COL in adata_sub.obs.columns:
    sc.pl.dotplot(
        adata_sub, var_names=_avail, groupby=LEIDEN_COL,
        dendrogram=True, standard_scale="var",
        title=f"Canonical marker dotplot (subset, {LEIDEN_COL})",
        show=False,
    )
    plt.savefig("results/figures/06c_subset_dotplot.png", dpi=150, bbox_inches="tight")
    plt.close()
    print("  标记物 dotplot 已保存: results/figures/06c_subset_dotplot.png")

# ---- 方法 2: 基因集评分（scanpy score_genes）----
_score_cols = []
for ct, gene_list in _markers.items():
    _present = [g for g in gene_list if g in adata_sub.var_names]
    if len(_present) < 2:
        continue
    col = f"score_{ct}"
    sc.tl.score_genes(
        adata_sub, gene_list=_present, score_name=col,
        ctrl_size=max(1, min(len(_present), 50)),
    )
    _score_cols.append(col)
print(f"  基因集评分: {len(_score_cols)} 个评分列 -> obs")

# === mLLMCelltype 多模型共识注释（子集）===
# B4：调用签名与 06 统一（marker_genes + species 模式，非 adata + cluster_key）
# B7：默认多模型共识，单模型时诚实降级
if MLLM_ENABLED and _patch_ok:
    try:
        from mllmcelltype import interactive_consensus_annotation

        _api_keys, _base_urls, _model_list = build_mllmcelltype_config(
            project_root=_root,
            model_list_override=MLLM_MODELS,
        )

        if _model_list:
            _n_models = len(_model_list)
            _use_discussion = _n_models >= 2

            if not _use_discussion:
                print(f"\u26a0\ufe0f 仅配置 {_n_models} 个模型，多模型共识降级为单模型注释")
                print(f"  -> 单一模型标签未经交叉验证，置信度低于多模型共识")
                print(f"  -> 如需多模型共识，请在 .env 中配置多个 LLM_GROUP* 或同一 group 的多档模型")

            print(f"mLLMCelltype 模型: {_model_list}")
            print(f"开始子集注释（{_n_models} 模型，{'讨论模式' if _use_discussion else '单模型模式'}）...")

            # B4：统一调用签名——marker_genes + species 模式（与 06 一致）
            # DEG for mLLMCelltype
            if "rank_genes_06c" not in adata_sub.uns:
                sc.tl.rank_genes_groups(
                    adata_sub, groupby=LEIDEN_COL, method="wilcoxon",
                    n_genes=30, key_added="rank_genes_06c",
                )

            _rgg = adata_sub.uns["rank_genes_06c"]
            _cluster_names = list(_rgg["names"].dtype.names)
            _marker_genes = {}
            for cl in _cluster_names:
                _marker_genes[str(cl)] = _rgg["names"][cl][:10].tolist()

            _result = interactive_consensus_annotation(
                marker_genes=_marker_genes,
                species="human",
                models=_model_list,
                api_keys=_api_keys,
                base_urls=_base_urls if _base_urls else None,
                tissue=_tissue_context,  # B4：子集实际组织上下文
                consensus_threshold=MLLM_CONSENSUS_THRESHOLD,
                entropy_threshold=MLLM_ENTROPY_THRESHOLD,
                max_discussion_rounds=(
                    MLLM_MAX_DISCUSSION_ROUNDS if _use_discussion else 0
                ),
                use_cache=False,
            )

            _llm_col = f"cell_type_llm_subset_{OUTPUT_VERSION}"
            if hasattr(_result, "cell_types") and _result.cell_types:
                _llm_map = _result.cell_types
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_llm_map)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成: {len(_llm_map)} 簇 -> obs['{_llm_col}']")
            elif isinstance(_result, dict) and "consensus" in _result:
                _consensus = _result["consensus"]
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_consensus)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成: {len(_consensus)} 簇 -> obs['{_llm_col}']")
            elif isinstance(_result, dict):
                adata_sub.obs[_llm_col] = (
                    adata_sub.obs[LEIDEN_COL].astype(str).map(_result)
                ).astype("category")
                print(f"mLLMCelltype 子集注释完成 (dict): {len(_result)} 簇 -> obs['{_llm_col}']")
            else:
                print(f"mLLMCelltype 返回格式异常: {type(_result).__name__}")
        else:
            print("无可用模型，跳过 mLLMCelltype（.env 未配置 LLM_GROUP*）")
    except ImportError:
        print("mLLMCelltype 未安装 (pip install mllmcelltype)")
    except Exception as e:
        print(f"mLLMCelltype 异常: {e}")
        import traceback
        traceback.print_exc()
else:
    if not MLLM_ENABLED:
        print("MLLM_ENABLED=False，跳过")
    else:
        print("mLLMCelltype patch 失败，跳过 LLM 注释")

_llm_col = f"cell_type_llm_subset_{OUTPUT_VERSION}"
if _llm_col not in adata_sub.obs.columns:
    adata_sub.obs[_llm_col] = pd.Categorical([np.nan] * adata_sub.n_obs)


### PI 拍板子集精细标签

PI 查看上方 dotplot + LLM 共识结果后，为子集的每个簇填入精细细胞类型标签。
这些标签将回流到主 adata 的 `cell_type_final_subset_v1` 列。

**命名建议**：精细标签应比全局标签更具体——
如全局标签 `T_cell` → 精细标签 `CD4_Tcm`、`CD8_Tem`、`Treg` 等。

In [ ]:
# === PI 拍板子集精细标签 ===
# PI 根据 dotplot + LLM 共识 + 基因集评分，为子集每个簇填入精细标签。
# 为什么精细标签与全局标签分开列？这是 SPEC 835 的层级粒度设计：
# cell_type_final_v1 保持大类标签（跨病种可比），
# cell_type_final_subset_v1 提供精细亚型（仅在子集内有意义）。
pi_subset_decisions = {
    # "0": "CD4_Tcm",
    # "1": "CD8_Tem",
    # "2": "Treg",
    # ... PI 逐簇填入
}

# B9：输出列名版本号与 06_annotated 的 OUTPUT_VERSION 一致
_final_subset_col = f"cell_type_final_subset_{OUTPUT_VERSION}"

if pi_subset_decisions and LEIDEN_COL in adata_sub.obs.columns:
    adata_sub.obs[_final_subset_col] = (
        adata_sub.obs[LEIDEN_COL].astype(str).map(pi_subset_decisions)
    )
    n_assigned = adata_sub.obs[_final_subset_col].notna().sum()
    print(f"子集精细标注: {n_assigned}/{adata_sub.n_obs} 细胞已标注 -> obs['{_final_subset_col}']")
    print(f"  标签种类: {adata_sub.obs[_final_subset_col].nunique()}")
else:
    adata_sub.obs[_final_subset_col] = np.nan
    adata_sub.obs[_final_subset_col] = (
        adata_sub.obs[_final_subset_col].astype("category")
    )
    print(f"PI 暂未填写子集精细标签，已预建 {_final_subset_col} 空列")


## 标签回流：将子集精细标签写回主 adata

这是 06c 最关键的一步。子集的精细标签只有回流到主 adata 的
`cell_type_final_subset_v1` 列后，下游 07 模块才能同时利用两个层级：

- `cell_type_final_v1`：大类标签（所有细胞都有，适合跨病种比较）
- `cell_type_final_subset_v1`：精细标签（仅子集细胞有值，其余 NaN）

**为什么不动 `cell_type_final_v1`？**
这是 SPEC 835 的硬约束——两个粒度的标签共存而不互相污染。
如果一个分析只需要大类，用 `cell_type_final_v1`；
如果需要在 T 细胞内部做精细 DEG，可以用 `cell_type_final_subset_v1` 过滤 NaN。

**为什么主 adata 回写版本号递增（v1→v2）？**
回流修改了主 adata 的 obs 列——这是一个新的数据状态，
应按版本化约定创建新版本而非覆盖原版。下游 notebook 指定
合适的版本号即可。

In [ ]:
# 标签回流：将子集精细标签写回主 adata。
print("=== 标签回流 ===")

# 重新加载主 adata（上游 path——而非更新后的版本，避免循环）
print(f"加载主 adata: {UPSTREAM_PATH}")
main_adata = sc.read_h5ad(UPSTREAM_PATH)

# 回流前快照：保存 cell_type_final_v1 用于回流后真实验证
# 为什么先存快照？只有 before/after 比对才能真正确认未修改——
# 不能靠读取一个从未写入的 uns key 冒充验证。
_check = main_adata.obs[f"cell_type_final_{OUTPUT_VERSION}"].copy()

# 初始化 cell_type_final_subset_v1 列——全部 NaN
main_adata.obs[f"cell_type_final_subset_{OUTPUT_VERSION}"] = np.nan

# 对齐：子集细胞 → 主 adata 中的对应细胞
# 为什么用 index 对齐？保证一一对应——子集是从主 adata copy 出来的，
# 原始 index 与主 adata 相同。
_sub_labels = adata_sub.obs[f"cell_type_final_subset_{OUTPUT_VERSION}"]
_common_idx = main_adata.obs_names.intersection(adata_sub.obs_names)
print(f"可回流细胞: {len(_common_idx):,} / {adata_sub.n_obs:,} 子集细胞")

# 将精细标签写入主 adata 对应细胞
# 为什么不用 .values？pandas 自动按 index 对齐赋值——
# .values 剥掉 index 后变成裸 numpy array，依赖顺序一致（脆弱）。
main_adata.obs.loc[_common_idx, f"cell_type_final_subset_{OUTPUT_VERSION}"] = (
    _sub_labels.loc[_common_idx]
)

# 处理缺失（子集中有但主 adata 中没有的细胞——不应发生，但防御处理）
_missing = set(adata_sub.obs_names) - set(_common_idx)
if _missing:
    print(f"⚠ 警告: {len(_missing)} 个子集细胞不在主 adata 中，无法回流")

_n_reflowed = main_adata.obs[f"cell_type_final_subset_{OUTPUT_VERSION}"].notna().sum()
print(f"已回流: {_n_reflowed:,} 细胞 -> obs['cell_type_final_subset_v1']")
print(f"  精细标签种类: {main_adata.obs[f'cell_type_final_subset_{OUTPUT_VERSION}'].nunique()}")

# 确认 cell_type_final_v1 未被修改——before/after 真实验证
# 为什么用 assert？只在回流前后做逐元素比对，不依赖从未写入的外部状态。
assert (_check == main_adata.obs[f"cell_type_final_{OUTPUT_VERSION}"]).all(), (
    "cell_type_final_v1 在回流过程中被意外修改！"
)
print("  cell_type_final_v1 保持不变（before/after 真实验证通过）")

# 记录回流元数据
main_adata.uns[f"cell_type_final_subset_{OUTPUT_VERSION}_notes"] = {
    "source": "06c_subset",
    "subset_filter": SUBSET_FILTER,
    "subset_h5ad": OUTPUT_PATH,
    "n_cells_refined": int(_n_reflowed),
    "refined_cell_types": sorted(
        main_adata.obs[f"cell_type_final_subset_{OUTPUT_VERSION}"].dropna().unique().tolist()
    ),
    "note": (
        "cell_type_final_v1 retains broad labels (all cells); "
        "cell_type_final_subset_v1 provides fine labels (subset cells only, NaN elsewhere). "
        "Downstream analyses choose the appropriate column."
    ),
    "timestamp": datetime.datetime.now().isoformat(),
}

In [ ]:
# === 构建统一精细标签列 ===
# 将 cell_type_final_v1（粗标签）和 cell_type_final_subset_v1（精细标签）合并为一列
# 规则：有 subset 精细标签的细胞用精细标签，其余用粗标签
_unified_col = f"cell_type_unified_{OUTPUT_VERSION}"
main_adata.obs[_unified_col] = main_adata.obs[f"cell_type_final_{OUTPUT_VERSION}"].copy()

# 用 subset 精细标签覆盖（仅对子集细胞）
_has_subset = main_adata.obs[f"cell_type_final_subset_{OUTPUT_VERSION}"].notna()
main_adata.obs.loc[_has_subset, _unified_col] = main_adata.obs.loc[_has_subset, f"cell_type_final_subset_{OUTPUT_VERSION}"]

_n_refined = _has_subset.sum()
print(f"✓ 统一标签列 '{_unified_col}' 已创建")
print(f"  {_n_refined:,} 细胞使用 subset 精细标签")
print(f"  {(~_has_subset).sum():,} 细胞保持粗标签")
print(f"  → 下游分析（07_downstream）可直接使用此列作为 groupby")

## 写出产物

两个 h5ad 文件：
1. **子集 h5ad**（`OUTPUT_PATH`）：子集细胞的完整分析结果，可独立用于子集下游分析
2. **主 adata 更新版**（`MAIN_OUTPUT_PATH`）：含 `cell_type_final_subset_v1` 列的主图谱新版本

两个文件命名均遵循版本化约定。

In [ ]:
# 写出两个 h5ad 产物。
print("=== 写出产物 ===")

# 1. 子集 h5ad
# 内存纪律：确认 sparse + dtype
assert sp.issparse(adata_sub.X) and adata_sub.X.dtype == np.float32, (
    f"adata_sub.X 不变量被破坏: sparse={sp.issparse(adata_sub.X)}, dtype={adata_sub.X.dtype}"
)

# 统一追踪字段——子集对象的独立追踪层（与 pipeline 编号命名一致）
adata_sub.uns["stage"] = "06c_subset"     # 本 stage 标识
adata_sub.uns["version"] = OUTPUT_VERSION                 # 与 OUTPUT_PATH 版本号一致
adata_sub.uns["upstream"] = [UPSTREAM_PATH]     # list 形式，支持多上游合并
adata_sub.uns["status"] = "experimental"        # PI 审查后改为 "promoted"

# 子集细节记录（嵌套 dict——细节层，与顶层追踪字段并存）
adata_sub.uns[f"06c_subset_{OUTPUT_VERSION}"] = {
    "upstream": UPSTREAM_PATH,
    "subset_filter": SUBSET_FILTER,
    "stages_rerun": ["03_hvg", "04_embedding", "05_clustering", "06_annotation"],
    "leiden_col_used": LEIDEN_COL,
    "timestamp": datetime.datetime.now().isoformat(),
}

adata_sub.write_h5ad(OUTPUT_PATH, compression="lzf")
print(f"子集 h5ad 已写出: {OUTPUT_PATH}")
print(f"  大小: {os.path.getsize(OUTPUT_PATH):,} bytes")

# 2. 主 adata 更新版（含 cell_type_final_subset_v1）
# main_adata 是 06 的产物，本 notebook 仅做回流更新（添加 cell_type_final_subset_v1 列）。
# 不放顶层 stage/version/upstream 字段——那会篡改 06 的溯源链。
# 主对象的追踪字段由 06_annotated 负责维护，此处仅记录本次回流操作的嵌套 dict。
main_adata.uns[f"06c_subset_reflow_{OUTPUT_VERSION}"] = {
    "upstream": UPSTREAM_PATH,
    "subset_h5ad": OUTPUT_PATH,
    "reflow_column": "cell_type_final_subset_v1",
    "note": "This version adds cell_type_final_subset_v1 from 06c subset re-analysis.",
    "timestamp": datetime.datetime.now().isoformat(),
}

main_adata.write_h5ad(MAIN_OUTPUT_PATH, compression="lzf")
print(f"\n主 adata 更新版已写出: {MAIN_OUTPUT_PATH}")
print(f"  大小: {os.path.getsize(MAIN_OUTPUT_PATH):,} bytes")
print(f"  含 cell_type_final_subset_v1: {'cell_type_final_subset_v1' in main_adata.obs.columns}")
print(f"  含 cell_type_final_v1: {'cell_type_final_v1' in main_adata.obs.columns}")

### 下一步：对子集内精细簇做深度剖析

06c 完成了 subset 重聚类和注释。如需对 subset 内的每个精细簇做逐簇深度解读（DEG、邻居对比、LLM 叙述），请运行 `06b_per_cluster.ipynb` 的 **subset 模式**。


In [ ]:
# === 下一步引导 ===
print("===== 06c 完成 → 下一步建议 =====\n")
print(f"子集产出: {OUTPUT_PATH}")
print(f"子集注释列: cell_type_final_subset_v1")
print(f"子集包含 {adata_sub.n_obs:,} cells, {adata_sub.obs[LEIDEN_COL].nunique()} clusters\n")
print("如需对这些精细簇做逐簇深度剖析（DEG + 邻居对比 + LLM 叙述），")
print("请打开 06b_per_cluster.ipynb，改 PARAMS 为 subset 模式:\n")
print(f'    UPSTREAM_PATH = "{OUTPUT_PATH}"')
print(f'    LABEL_COL = "cell_type_final_subset_v1"')
print(f'    MODE = "subset"')
print(f'    OUTPUT_DIR = "results/figures/06b_per_cluster_subset"\n')
print("然后 Run All。06b 会对 subset 内的每个精细簇产出独立报告。")


### Stage 06c Verdict

本 stage 完成后应确认：
- [ ] subset 精细注释完成（cell_type_final_subset_v1 已填入）
- [ ] 统一标签列 cell_type_unified_v1 已生成
- [ ] 如需对 subset 内簇做深度剖析 -> 转 06b(subset 模式)


In [ ]:
# 内存纪律——del + gc 释放跨越 stage 边界。
# 为什么必须释放？子集 adata 虽比主 adata 小，但连同 embedding
# + obsm 矩阵仍可达到数 GB。不及时释放会累积到下游 OOM。
del adata_sub
del main_adata
gc.collect()
print("内存已释放")